### Baifern Detection Model

Make sure we've got the latest version of fastai:

In [1]:
#!pip install -U fastai

In [2]:
from fastai.vision.all import *

Copy path from input dataset

In [3]:
path = Path('/kaggle/input/datasets/nohaxjusttul/baifern-dataset')

Define custom rule for labeling (for nested folder in NotBaifern)

In [4]:
def get_label(file_path):
    if 'NotBaifern' in str(file_path):
        return 'NotBaifern'
    else:
        return 'Baifern'

Create a clean list of files by removing the corrupted ones

In [5]:
print("Scanning dataset for corrupted files...")
all_files = get_image_files(path)
failed_files = verify_images(all_files)
print(f"Found {len(failed_files)} corrupted files!")
clean_files = L([f for f in all_files if f not in failed_files])
print(f"Proceeding with {len(clean_files)} clean images.")

Scanning dataset for corrupted files...
Found 1 corrupted files!
Proceeding with 252 clean images.


Use from_path_func, so we can use clean_files

In [6]:
dls = ImageDataLoaders.from_path_func(
    path, 
    clean_files,
    valid_pct=0.2, 
    seed=42,               # Keeps the validation split consistent across runs
    label_func=get_label,  # Apply our custom rule
    item_tfms=Resize(224)
)

Train the model

In [7]:
learn = vision_learner(dls, resnet18, metrics=error_rate)
learn.fine_tune(5)

epoch,train_loss,valid_loss,error_rate,time
0,1.094885,2.644218,0.500000,00:32


epoch,train_loss,valid_loss,error_rate,time
0,0.456335,1.681993,0.400000,00:39
1,0.333971,0.963907,0.300000,00:40
2,0.242730,0.411624,0.140000,00:40
3,0.182405,0.186142,0.060000,00:44
4,0.149444,0.101062,0.040000,01:00


Export model

In [10]:
learn.export('/kaggle/working/model.pkl')